# Dates and periods in pandas

Python stage, lesson P17. Part of [datafromzero](https://datafromzero.github.io), data and text CC BY 4.0.

**Goals:** parse text into dates with `to_datetime` and an explicit format; use the `dt` accessor and filter by a date range; roll up by month with `resample`; compare each month with the previous one using `shift`.

**How to use this notebook:** run the cells from top to bottom. When in doubt, **Run all from the top**: restart the session and run every cell from the first.
Every exercise has a cell for your code and a check cell that compares your answer with a known answer from the cafe-sales data. Hints and solutions are folded away
under each exercise, free to open. The working cell of Exercise 2 starts empty: you write it.

## Setup

In [ ]:
import math
import numbers

import numpy as np
import pandas as pd

# Run this cell as it is. It also holds the answer checker used by the exercise cells.
EXERCISES = ["Exercise 1", "Exercise 2", "Exercise 3"]


RESULTS = {}


def same(got, expected):
    """True when two values agree: floats within rounding, lists and dicts item by item."""
    try:
        if isinstance(expected, dict):
            return isinstance(got, dict) and got.keys() == expected.keys() and all(same(got[k], expected[k]) for k in expected)
        if isinstance(expected, (list, tuple)):
            return isinstance(got, (list, tuple)) and len(got) == len(expected) and all(same(g, e) for g, e in zip(got, expected))
        if isinstance(expected, bool) or isinstance(got, bool):
            return type(got) is type(expected) and got == expected
        if isinstance(expected, float):
            return isinstance(got, numbers.Real) and math.isclose(got, expected, rel_tol=1e-9, abs_tol=1e-9)
        if isinstance(expected, int):
            return isinstance(got, numbers.Integral) and got == expected
        return type(got) is type(expected) and got == expected
    except Exception:
        return False


def check(label, name, expected, traps=None):
    """Compare the variable called `name` with the known answer. It never stops Run all."""
    got = globals().get(name)
    if got is None:
        RESULTS[label] = None
        print(f"{label}: give {name} a value, then run this cell again.")
        return False
    if same(got, expected):
        RESULTS[label] = True
        print(f"{label}: correct.")
        return True
    RESULTS[label] = False
    print(f"{label}: not yet. Expected {expected!r}, but {name} is {got!r}.")
    for wrong, message in (traps or {}).items():
        if same(got, wrong):
            print("Hint:", message)
    return False

## Data

In [ ]:
# The five cafe-sales tables (version 1), read from the release tag v0.1.0.
BASE = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.1.0/datasets/cafe-sales/v1/"
stores = pd.read_csv(BASE + "stores.csv")
customers = pd.read_csv(BASE + "customers.csv")
products = pd.read_csv(BASE + "products.csv")
orders = pd.read_csv(BASE + "orders.csv")
order_items = pd.read_csv(BASE + "order_items.csv")

# Known answer from the dataset README: 3, 150, 16, 1,800 and 3,287 rows.
assert (len(stores), len(customers), len(products), len(orders), len(order_items)) == (3, 150, 16, 1800, 3287)
print("Data loaded: 3 stores, 150 customers, 16 products, 1,800 orders, 3,287 order lines.")

## Unit 1: to_datetime: name the format

**Predict first.** `ordered_at` is text. What are the first and the last order times after the conversion, and how many dates are missing?

In [ ]:
orders = orders.assign(when=pd.to_datetime(orders["ordered_at"], format="%Y-%m-%d %H:%M:%S"))
print(orders["when"].min(), orders["when"].max())
print(orders["when"].isna().sum())

**Predict first.** read day first, which month does each text give? Read month first, with `errors="coerce"`, how many dates are missing?

In [ ]:
typed = pd.Series(["03/04/2026", "13/04/2026", "05/11/2026"])
as_days = pd.to_datetime(typed, format="%d/%m/%Y")
as_months = pd.to_datetime(typed, format="%m/%d/%Y", errors="coerce")
print(as_days.dt.month.tolist())
print(as_months.isna().sum())

## Unit 2: dt and date ranges: parts, and the last day

**Predict first.** which day of the week has the most orders, and which the fewest?

In [ ]:
weekday = orders["when"].dt.day_name()
print(weekday.value_counts())

**Predict first.** how many orders does each version keep?

In [ ]:
first_quarter = orders[(orders["when"] >= "2026-01-01") & (orders["when"] <= "2026-03-31")]
whole_quarter = orders[(orders["when"] >= "2026-01-01") & (orders["when"] < "2026-04-01")]
print(len(first_quarter), len(whole_quarter))

## Unit 3: resample: one row for each month

**Predict first.** how many rows will `monthly` have, and will its numbers add up to 1,800?

In [ ]:
monthly = orders.set_index("when").resample("MS")["order_id"].size()
print(monthly)
print(monthly.sum())

**Predict first.** what will each version list for the months in which customer 5 did not order?

In [ ]:
one = orders[orders["customer_id"] == 5]
by_resample = one.set_index("when").resample("MS")["order_id"].size()
by_month = one.groupby(one["when"].dt.month).size()
print(by_resample.tolist())
print(by_month.to_dict())

## Unit 4: shift: the month before

**Predict first.** what will the first value be, and in which month did the orders grow the most?

In [ ]:
change = monthly - monthly.shift(1)
print(change.tolist())

## Unit 5: Check: two routes to the same months

**Predict first.** do `groupby` on the month number and `resample` give the same six counts?

In [ ]:
by_number = orders.groupby(orders["when"].dt.month).size()
print(by_number.tolist() == monthly.tolist())

## Finish it

### Exercise 1: Finish it

**Task:** How many orders did the cafes take from January 1 to May 31, both days included? Replace the first `None` with the orders in that range, using the column `when` of unit 1, then set `answer_1` to the number of rows.

**Expected:** one whole number.

In [ ]:
first_five = None  # replace None: the orders of when from "2026-01-01" to "2026-05-31", both days included
answer_1 = None  # then set answer_1 to the number of rows of first_five

In [ ]:
_ = check("Exercise 1", "answer_1", 1447, traps={1434: "1434 leaves out the orders of the afternoon of May 31: \"2026-05-31\" means midnight at the start of that day. Stop before the first moment of June."})

<details>
<summary>Hint</summary>

Unit 2 keeps a range of days. A date written alone means the start of that day, so how should the range end?

</details>

<details>
<summary>Solution</summary>

```python
first_five = orders[(orders["when"] >= "2026-01-01") & (orders["when"] < "2026-06-01")]
answer_1 = len(first_five)
```

The answer is 1447: 242, 218, 258, 349 and 380 orders for January to May. Ending the range with `<= "2026-05-31"` gives 1434, because the orders of May 31 after midnight are cut off, and the check cell says so.

</details>

## On your own

### Exercise 2: On your own

**Task:** What was the largest rise in the number of orders of Old Town (store 1) from one month to the next? Before you start, say in one sentence what kind of number the answer is. Count the orders of store 1 for each month and compare each month with the one before. Write the code in the empty cell below and set `answer_2`.

**Expected:** one decimal number.

In [ ]:
_ = check("Exercise 2", "answer_2", 30.0, traps={91.0: "91 is the largest rise of all three cafes together, the number of unit 4. Keep only the orders of store 1 before you resample."})

<details>
<summary>Hint</summary>

Unit 3 counts the orders of each month and unit 4 compares a month with the one before. Which rows do you keep first?

</details>

<details>
<summary>Solution</summary>

```python
old_town = orders[orders["store_id"] == 1]
monthly_old_town = old_town.set_index("when").resample("MS")["order_id"].size()
answer_2 = (monthly_old_town - monthly_old_town.shift(1)).max()
```

The answer is 30.0: Old Town had 103 orders in February and 133 in March. Without the filter on the store the largest rise is 91, April for all cafes together, and the check cell says so. The answer is a decimal because the `nan` of the first month makes the whole column decimals.

</details>

## Check yourself

### Exercise 3: Check yourself: spot the flaw

**Task:** Spot the flaw. The bakery's supplier typed the delivery dates in two spellings in one column. A colleague parses the column and counts the deliveries in March, and the answer looks too small. Fix the parse, then set `answer_3 = (parsed.dt.month == 3).sum()`.

**Expected:** one whole number.

In [ ]:
supplier = pd.Series(["2026-03-02", "15/03/2026", "2026-03-28", "03/04/2026", "2026-04-04"])  # two spellings, the slash dates day first
parsed = pd.to_datetime(supplier, format="%Y-%m-%d", errors="coerce")  # the colleague's line
answer_3 = None  # fix the line above, then set answer_3 = (parsed.dt.month == 3).sum()

In [ ]:
_ = check("Exercise 3", "answer_3", 3, traps={2: "Only 2 deliveries are read as March: the slash dates became NaT, because one format reads one spelling. Print parsed.isna().sum(), then read each spelling with its own format."})

<details>
<summary>Hint</summary>

Print `parsed.isna().sum()`: two dates are missing, and they are the ones with slashes. One format reads one spelling. Read each spelling with its own format, then fill the gaps of one result from the other, as in lesson P16.

</details>

<details>
<summary>Solution</summary>

```python
supplier = pd.Series(["2026-03-02", "15/03/2026", "2026-03-28", "03/04/2026", "2026-04-04"])
by_iso = pd.to_datetime(supplier, format="%Y-%m-%d", errors="coerce")
by_slash = pd.to_datetime(supplier, format="%d/%m/%Y", errors="coerce")
parsed = by_iso.fillna(by_slash)
answer_3 = (parsed.dt.month == 3).sum()
```

The answer is 3: 2, 15 and 28 March. The slash dates are day first, so `03/04/2026` is 3 April. Read with the first format alone, the two slash dates became `NaT` and the count was 2; the check cell says so.

</details>

## Run all from the top

Restart the session, run every cell from the first, then read this cell last.

In [ ]:
for label in EXERCISES:
    state = RESULTS.get(label)
    print(f"{label}: " + ("correct" if state else "not yet" if state is False else "not started"))
print(f"{sum(1 for label in EXERCISES if RESULTS.get(label))} of {len(EXERCISES)} exercises correct.")